# Label_Generation_QC_v1

**Goal:** Generate and visually verify clean 4-class training labels from scratch, before any patch generation or model training.

**Fresh-start architecture** (legacy debt from v6/v7/v8 shed):

| Stage | Output |
|---|---|
| Droplet detection | NPC watershed (histogram-clipped) → distance-transform markers → 10 px erosion |
| Nucleus detection | Adaptive threshold within droplet → per-droplet Otsu fallback → may be empty |
| NPC puncta | Nucleus-boundary-relative annular zone (raw, unclipped NPC) |
| Label representation | 4-channel **multi-label** binary stack (classes overlap) |

| Dim order | `(T, Z, C, Y, X)` |
|---|---|
| Channels | `C=0 Membrane · C=1 NLS · C=2 NPC` |
| Classes | `0 Background · 1 Droplet · 2 NPC · 3 Nucleus` |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
import tifffile
from pathlib import Path

from skimage import filters, morphology, segmentation, measure
from skimage.feature import peak_local_max
from scipy import ndimage

rng = np.random.default_rng(42)  # deterministic sampling for QC displays

In [ ]:
# ── Paths ─────────────────────────────────────────────────────────────────────
HYPERSTACK_PATH = Path("/path/to/control_extract_1.1.tif")   # <-- EDIT
OUTPUT_DIR      = Path("/path/to/patches")                   # <-- EDIT

# ── Channel indices  (T, Z, C, Y, X) ──────────────────────────────────────────
CH_MEMBRANE = 0
CH_NLS      = 1
CH_NPC      = 2

# ── 4-class scheme (multi-label: channels may overlap) ─────────────────────────
CLASS_BACKGROUND = 0
CLASS_DROPLET    = 1
CLASS_NPC        = 2
CLASS_NUCLEUS    = 3
CLASS_NAMES      = ["Background", "Droplet", "NPC", "Nucleus"]

# ── Calibration (control_extract_1.1.tif, carried over from v7) ────────────────
PIXEL_UM = 0.108                       # µm / px  (50,000 px ≈ 583 µm²)
def um2_to_px(um2): return um2 / (PIXEL_UM ** 2)

# ── Droplet detection — NPC watershed ──────────────────────────────────────────
# Histogram clip is applied to a COPY of the NPC plane ONLY for droplet detection.
# It suppresses bright NPC puncta so they cannot fragment the watershed.
# The raw NPC channel is re-pulled fresh for NPC puncta detection (see Stage 3).
NPC_CLIP_LO_PCT   = 1.0     # lower clip percentile
NPC_CLIP_HI_PCT   = 80.0    # upper clip percentile (v7 used p1–p80)
BLUR_SIGMA        = 8.0     # Gaussian blur on clipped image (v7 calibration)
ADAPTIVE_BLOCK    = 301     # local-threshold block size for droplet foreground (odd)
ADAPTIVE_OFFSET   = -0.05   # local-threshold offset (v7 calibration)
MIN_PEAK_DISTANCE = 40      # min px between watershed seeds  ← KEY tuning knob
MIN_DROPLET_AREA  = um2_to_px(150)   # ≈ 12,860 px ; reject small debris
DROPLET_MIN_CIRC  = 0.70    # reject merged / kidney-bean regions (v7)
EROSION_PX        = 10       # erode each watershed region edge inward (v7)

# ── NPC puncta — nucleus-boundary-relative ─────────────────────────────────────
NPC_MARGIN_PX = 5      # half-width of annular zone straddling the nucleus edge
NPC_STD_MULT  = 2.0    # threshold = mean + NPC_STD_MULT*std of RAW NPC interior (v7)

# ── Nucleus — adaptive with Otsu fallback ──────────────────────────────────────
NUCLEUS_BLOCK_SIZE   = None    # None = auto (~1/3 droplet diameter, forced odd)
NUCLEUS_MIN_FRACTION = 0.01    # below → treat as no nucleus (early timepoints)
NUCLEUS_MAX_FRACTION = 0.50    # above → threshold ran away (v7 max_droplet_fraction)
NUCLEUS_MIN_CIRC     = 0.40    # circularity gate (v7)

# ── Patch generation ───────────────────────────────────────────────────────────
PATCH_SIZE = 512
Z_FLOOR    = 6     # exclude coverslip-artifact planes below this z
N_PREVIEW  = 10    # patches to spot-check in Stage 5

In [ ]:
def load_hyperstack(path: Path) -> np.ndarray:
    """Load the hyperstack TIFF as (T, Z, C, Y, X)."""
    hs = tifffile.imread(str(path))
    assert hs.ndim == 5, f"Expected 5D (T,Z,C,Y,X), got {hs.ndim}D: {hs.shape}"
    print(f"Loaded: shape={hs.shape}  dtype={hs.dtype}")
    return hs


hyperstack = load_hyperstack(HYPERSTACK_PATH)
T, Z, C, Y, X = hyperstack.shape
print(f"T={T}  Z={Z}  C={C}  Y={Y}  X={X}")

---
## Methodology review — issues checked & decisions

Four potential methodology problems were identified and addressed in this design. Read before running.

### 1. Histogram isolation (the core fix)
The NPC channel is used for **two different purposes** with **opposite** intensity requirements:

| Purpose | Needs | Treatment |
|---|---|---|
| Droplet detection | puncta *suppressed* so they don't fragment the watershed | clip p1–p80 |
| NPC puncta detection | puncta *preserved* — they are the signal | **raw, unclipped** |

If a single clipped array were shared between both steps, NPC detection would read a signal with its puncta already clamped away — silently destroying the exact class it is trying to label. This is a plausible root cause of the total NPC failure you saw.

**Enforcement:** every channel access goes through `extract_plane()`, which returns a fresh `float32` **copy** pulled from the original hyperstack. `clip_histogram()` never writes in place. Droplet detection clips its own local copy; NPC detection re-pulls the raw plane. A QC cell below asserts the source hyperstack is byte-for-byte unchanged after a full detection pass.

### 2. Label representation — multi-label, not integer
Your label diagnostic shows four overlapping binary channels (the nucleus footprint also belongs to the droplet). A single integer plane with paint-order overwrite would erase the droplet class under every nucleus. Labels are therefore stored as a `(4, H, W)` binary stack where channels overlap. `collapse_to_integer()` is provided if you later want an argmax map for a softmax head — **confirm which your model expects.**

### 3. NPC detection requires a detected nucleus
NPC puncta are anchored to the nucleus boundary (your choice). Consequence: at early timepoints with NPC puncta but no sealed NLS-positive nucleus, no NPC will be labeled. Acceptable if confirmed-nucleus NPC is the training target — **flag if you need pre-nucleus NPC.**

### 4. Both watershed failure modes
- *Non-uniform interiors* (Image 1): each droplet region is `binary_fill_holes`-ed before erosion → solid class 1.
- *Internal structures labeled instead of boundary* (Image 2): markers come from the **distance transform of the clipped binary foreground** (shape), never from intensity, so an internal NPC ring cannot seed a basin. A circularity gate rejects merged blobs.

In [ ]:
# ── Histogram-safe channel access ──────────────────────────────────────────────
# These two functions are the guarantee behind methodology point #1.
# Nothing downstream is permitted to mutate the hyperstack in place.

def extract_plane(hyperstack: np.ndarray, t: int, z: int, c: int) -> np.ndarray:
    """
    Return a single 2D plane as a fresh float32 COPY.

    Always a copy: callers may clip/blur/threshold the result freely without
    ever touching the source hyperstack or any other class's view of the channel.
    """
    return np.asarray(hyperstack[t, z, c], dtype=np.float32).copy()


def clip_histogram(img: np.ndarray, lo_pct: float, hi_pct: float) -> np.ndarray:
    """
    Percentile-clip and 0–1 normalise a COPY of `img`. Never writes in place.

    Used ONLY for droplet detection to suppress NPC puncta. The returned array
    is a new object; `img` is unchanged.
    """
    work = img.astype(np.float32, copy=True)
    lo, hi = np.percentile(work, [lo_pct, hi_pct])
    work = np.clip(work, lo, hi)
    rng_ = hi - lo
    if rng_ <= 0:
        return np.zeros_like(work)
    return (work - lo) / rng_


def _circularity(region) -> float:
    """4*pi*area / perimeter^2. Returns 0 for degenerate perimeter."""
    p = region.perimeter
    if p <= 0:
        return 0.0
    return float(4.0 * np.pi * region.area / (p * p))

In [ ]:
# ── Histogram-isolation QC ─────────────────────────────────────────────────────
# Proves a full detection pass leaves the source channel untouched. Run anytime.

def assert_histogram_isolation(hyperstack: np.ndarray, t: int, z: int) -> None:
    before = hyperstack[t, z, CH_NPC].copy()

    raw      = extract_plane(hyperstack, t, z, CH_NPC)
    _clipped = clip_histogram(raw, NPC_CLIP_LO_PCT, NPC_CLIP_HI_PCT)  # for droplets
    raw_again = extract_plane(hyperstack, t, z, CH_NPC)               # for NPC puncta

    after = hyperstack[t, z, CH_NPC]
    assert np.array_equal(before, after), "SOURCE MUTATED — histogram bled across classes!"
    assert np.array_equal(raw, raw_again), "Re-pulled raw plane differs — copy failure!"
    assert not np.array_equal(_clipped, raw / (raw.max() + 1e-8)), "Clip had no effect?"
    print("Histogram isolation OK: source unchanged; raw NPC re-pull is identical & unclipped.")


assert_histogram_isolation(hyperstack, t=min(4, T - 1), z=min(8, Z - 1))

---
## Stage 1 — Core detection functions & single-droplet audit

Four core functions, then a single-droplet audit to confirm class assignment before any scale-up.

In [ ]:
def detect_droplets_npc_watershed(
    npc_plane: np.ndarray,
    clip_lo: float = NPC_CLIP_LO_PCT,
    clip_hi: float = NPC_CLIP_HI_PCT,
    blur_sigma: float = BLUR_SIGMA,
    block_size: int = ADAPTIVE_BLOCK,
    offset: float = ADAPTIVE_OFFSET,
    min_peak_distance: int = MIN_PEAK_DISTANCE,
    min_area: float = MIN_DROPLET_AREA,
    min_circ: float = DROPLET_MIN_CIRC,
    erosion_px: int = EROSION_PX,
) -> list[dict]:
    """
    Detect individual droplets from one NPC plane.

    Pipeline
    --------
    1. clip_histogram (p1–p80) on a COPY → suppresses bright puncta.
    2. Gaussian blur (sigma=8).
    3. Local adaptive threshold (block=301, offset=-0.05) → binary foreground.
    4. Clean: fill holes + remove small objects.
    5. Distance transform of the binary mask → peak_local_max seeds (SHAPE-based,
       immune to internal NPC intensity structure).
    6. Watershed on -distance, masked to foreground → separates touching droplets.
    7. Per region: fill holes, circularity + area gates, erode `erosion_px` inward.

    Returns
    -------
    list[dict]: {label, mask(bool YX, eroded+filled), bbox, centroid, area, circ}
    """
    img_norm = clip_histogram(npc_plane, clip_lo, clip_hi)           # copy, safe
    img_blur = filters.gaussian(img_norm, sigma=blur_sigma)

    local_thresh = filters.threshold_local(img_blur, block_size=block_size, offset=offset)
    fg = img_blur > local_thresh
    fg = ndimage.binary_fill_holes(fg)
    fg = morphology.remove_small_objects(fg, min_size=int(min_area))

    dist = ndimage.distance_transform_edt(fg)
    seeds = peak_local_max(dist, min_distance=min_peak_distance, labels=fg)
    markers = np.zeros(dist.shape, dtype=np.int32)
    for i, (r, c) in enumerate(seeds, start=1):
        markers[r, c] = i
    if markers.max() == 0:
        return []

    ws = segmentation.watershed(-dist, markers, mask=fg)

    droplets = []
    selem = morphology.disk(erosion_px)
    for region in measure.regionprops(ws):
        if region.area < min_area:
            continue
        circ = _circularity(region)
        if circ < min_circ:
            continue                                   # rejects merged/kidney shapes
        region_mask = ndimage.binary_fill_holes(ws == region.label)  # solid interior
        eroded = morphology.binary_erosion(region_mask, selem)       # 10 px inward
        if not eroded.any():
            continue
        ys, xs = np.where(eroded)
        droplets.append({
            "label":    int(region.label),
            "mask":     eroded,
            "bbox":     (int(ys.min()), int(xs.min()), int(ys.max()) + 1, int(xs.max()) + 1),
            "centroid": (float(ys.mean()), float(xs.mean())),
            "area":     int(eroded.sum()),
            "circ":     round(circ, 3),
        })
    return droplets

In [ ]:
def detect_nucleus_adaptive(
    nls_crop: np.ndarray,
    droplet_mask_crop: np.ndarray,
    block_size: int | None = NUCLEUS_BLOCK_SIZE,
    min_frac: float = NUCLEUS_MIN_FRACTION,
    max_frac: float = NUCLEUS_MAX_FRACTION,
    min_circ: float = NUCLEUS_MIN_CIRC,
) -> tuple[np.ndarray, str]:
    """
    Detect the nucleus interior from the NLS crop of one droplet.

    Primary: adaptive local threshold within the droplet mask.
    Fallback: per-droplet Otsu on NLS values inside the droplet.
    If neither passes the area + circularity gates → return an EMPTY mask, "none"
    (biologically correct for early timepoints with no sealed nucleus — we do not
    hallucinate a nucleus to satisfy a minimum).

    Returns (nucleus_mask bool, method in {"adaptive","otsu","none"}).
    """
    nls = nls_crop.astype(np.float32, copy=True)
    droplet_area = int(droplet_mask_crop.sum())
    empty = np.zeros_like(droplet_mask_crop, dtype=bool)
    if droplet_area == 0:
        return empty, "none"

    diameter = 2.0 * np.sqrt(droplet_area / np.pi)
    if block_size is None:
        bs = max(3, int(diameter / 3))
        bs = bs + 1 if bs % 2 == 0 else bs
    else:
        bs = block_size

    def _clean_and_gate(mask):
        mask = mask & droplet_mask_crop
        mask = morphology.remove_small_objects(mask, min_size=64)
        mask = ndimage.binary_fill_holes(mask)
        frac = mask.sum() / droplet_area
        if not (min_frac <= frac <= max_frac):
            return None
        lbl = measure.label(mask)
        props = measure.regionprops(lbl)
        if not props:
            return None
        biggest = max(props, key=lambda r: r.area)
        if _circularity(biggest) < min_circ:
            return None
        return lbl == biggest.label   # keep single largest nucleus

    # primary — adaptive
    try:
        local_t = filters.threshold_local(nls, block_size=bs)
        cand = _clean_and_gate(nls > local_t)
        if cand is not None:
            return cand, "adaptive"
    except Exception:
        pass

    # fallback — Otsu on interior values
    try:
        vals = nls[droplet_mask_crop]
        if vals.size and vals.max() > vals.min():
            cand = _clean_and_gate(nls > filters.threshold_otsu(vals))
            if cand is not None:
                return cand, "otsu"
    except Exception:
        pass

    return empty, "none"

In [ ]:
def detect_npc_puncta(
    npc_crop_raw: np.ndarray,
    nucleus_mask_crop: np.ndarray,
    droplet_mask_crop: np.ndarray,
    margin_px: int = NPC_MARGIN_PX,
    std_mult: float = NPC_STD_MULT,
) -> np.ndarray:
    """
    Detect NPC puncta using the nucleus boundary as the spatial anchor.

    `npc_crop_raw` MUST be the raw, unclipped NPC plane (re-pulled via
    extract_plane), NOT the clipped array used for droplet detection — otherwise
    the puncta signal is gone. This is methodology point #1.

    Strategy
    --------
    - Annular search zone straddling the nucleus edge:
          outer = dilate(nucleus, margin_px); inner = erode(nucleus, margin_px)
          zone  = outer & ~inner
      Excludes the nucleus interior (antibody-diffusion artifact) and the droplet
      wall (v6-era error) by construction.
    - Threshold = mean + std_mult*std of RAW NPC over the droplet interior (v7).
    - Puncta = (npc > threshold) within the zone.

    Returns NPC puncta mask (bool, crop coords). Empty if no nucleus to anchor to.
    """
    if nucleus_mask_crop.sum() == 0:
        return np.zeros_like(nucleus_mask_crop, dtype=bool)

    npc = npc_crop_raw.astype(np.float32, copy=True)
    selem = morphology.disk(margin_px)
    outer = morphology.binary_dilation(nucleus_mask_crop, selem)
    inner = morphology.binary_erosion(nucleus_mask_crop, selem)
    zone = outer & ~inner

    interior_vals = npc[droplet_mask_crop]
    if interior_vals.size == 0:
        return np.zeros_like(nucleus_mask_crop, dtype=bool)
    thresh = interior_vals.mean() + std_mult * interior_vals.std()

    return (npc > thresh) & zone

In [ ]:
def build_label_stack(
    shape: tuple,
    droplets: list[dict],
    nucleus_masks: list[np.ndarray],
    npc_masks: list[np.ndarray],
) -> np.ndarray:
    """
    Compose a 4-channel MULTI-LABEL binary stack (channels may overlap).

        ch0 Background = NOT droplet
        ch1 Droplet    = solid eroded droplet interior (INCLUDES nucleus footprint)
        ch2 NPC        = puncta on the nuclear envelope
        ch3 Nucleus    = nucleus interior

    Matches the per-channel label diagnostic (droplet solid, nucleus a subset of it).
    Returns uint8 array (4, H, W).
    """
    H, W = shape
    droplet = np.zeros((H, W), bool)
    npc     = np.zeros((H, W), bool)
    nucleus = np.zeros((H, W), bool)

    for d, nuc, npcm in zip(droplets, nucleus_masks, npc_masks):
        droplet |= d["mask"]
        r0, c0, r1, c1 = d["bbox"]
        nucleus[r0:r1, c0:c1] |= nuc
        npc[r0:r1, c0:c1] |= npcm

    droplet |= nucleus          # nucleus lies inside the droplet → droplet is solid
    background = ~droplet
    return np.stack([background, droplet, npc, nucleus]).astype(np.uint8)


def collapse_to_integer(label_stack: np.ndarray) -> np.ndarray:
    """
    Collapse the multi-label stack to a single integer map for a softmax head.
    Priority (highest wins): Nucleus > NPC > Droplet > Background.
    Use ONLY if your model expects mutually-exclusive classes — confirm first.
    """
    _, H, W = label_stack.shape
    out = np.zeros((H, W), dtype=np.uint8)
    out[label_stack[CLASS_DROPLET].astype(bool)]  = CLASS_DROPLET
    out[label_stack[CLASS_NPC].astype(bool)]      = CLASS_NPC
    out[label_stack[CLASS_NUCLEUS].astype(bool)]  = CLASS_NUCLEUS
    return out

In [ ]:
_LABEL_CMAP = ListedColormap(["#2d004b", "#1f78b4", "#ffd700", "#1b7837"])  # bg/drop/npc/nuc

def _norm(img):
    lo, hi = np.percentile(img, [1, 99])
    return np.clip((img - lo) / (hi - lo + 1e-8), 0, 1)


def audit_single_droplet(hyperstack, t: int, z: int, droplet_idx: int = 0) -> None:
    """
    STAGE 1 — run the full pipeline on one droplet and render a diagnostic.
    Row 1: raw NLS / NPC / Membrane (full plane, droplet boxed).
    Row 2: droplet mask / nucleus mask / NPC mask / 4-class label (crop).
    """
    nls = extract_plane(hyperstack, t, z, CH_NLS)
    npc = extract_plane(hyperstack, t, z, CH_NPC)   # raw — re-pulled for puncta
    mem = extract_plane(hyperstack, t, z, CH_MEMBRANE)

    droplets = detect_droplets_npc_watershed(npc)
    if not droplets:
        print(f"No droplets detected at t={t}, z={z}.")
        return
    droplets.sort(key=lambda d: d["area"], reverse=True)
    d = droplets[min(droplet_idx, len(droplets) - 1)]
    r0, c0, r1, c1 = d["bbox"]

    dm_crop  = d["mask"][r0:r1, c0:c1]
    nls_crop = nls[r0:r1, c0:c1]
    npc_crop = npc[r0:r1, c0:c1]
    nuc_crop, method = detect_nucleus_adaptive(nls_crop, dm_crop)
    npc_mask = detect_npc_puncta(npc_crop, nuc_crop, dm_crop)

    stack = build_label_stack(nls.shape, [d], [nuc_crop], [npc_mask])
    label_int = collapse_to_integer(stack)[r0:r1, c0:c1]

    fig, ax = plt.subplots(2, 4, figsize=(18, 9))
    for a in ax.ravel():
        a.axis("off")
    for col, (img, name) in enumerate([(nls, "Raw NLS"), (npc, "Raw NPC"), (mem, "Raw Membrane")]):
        ax[0, col].imshow(_norm(img), cmap="gray")
        ax[0, col].add_patch(plt.Rectangle((c0, r0), c1 - c0, r1 - r0, ec="red", fc="none", lw=1.5))
        ax[0, col].set_title(name)
    ax[0, 3].set_title(f"t={t} z={z}\ndroplet area={d['area']} circ={d['circ']}\nnucleus={method}")

    ax[1, 0].imshow(dm_crop, cmap="gray");           ax[1, 0].set_title("Droplet mask")
    ax[1, 1].imshow(nuc_crop, cmap="gray");          ax[1, 1].set_title(f"Nucleus ({method})")
    ax[1, 2].imshow(npc_mask, cmap="gray");          ax[1, 2].set_title("NPC puncta")
    ax[1, 3].imshow(label_int, cmap=_LABEL_CMAP, vmin=0, vmax=3); ax[1, 3].set_title("4-class label")
    plt.tight_layout(); plt.show()

In [ ]:
# ── Stage 1 Runner ─────────────────────────────────────────────────────────────
T_AUDIT, Z_AUDIT, IDX_AUDIT = 4, 8, 0   # mid-timecourse, focused plane, largest droplet
audit_single_droplet(hyperstack, t=T_AUDIT, z=Z_AUDIT, droplet_idx=IDX_AUDIT)

---
## Stage 2 — Watershed separation validation (dense timepoints)

Each detected region should be ~circular and individually separated. Merged/kidney-bean regions = watershed failure (and are now rejected by the circularity gate, so they show as *uncounted* rather than mislabeled).

In [ ]:
def validate_watershed(hyperstack, t_range: list[int], z: int) -> None:
    """STAGE 2 — one row per timepoint: clipped NPC / seeds+watershed / accepted droplets."""
    fig, ax = plt.subplots(len(t_range), 3, figsize=(15, 5 * len(t_range)))
    if len(t_range) == 1:
        ax = ax[None, :]
    for i, t in enumerate(t_range):
        npc = extract_plane(hyperstack, t, z, CH_NPC)
        clipped = clip_histogram(npc, NPC_CLIP_LO_PCT, NPC_CLIP_HI_PCT)
        droplets = detect_droplets_npc_watershed(npc)

        accepted = np.zeros(npc.shape, bool)
        for d in droplets:
            accepted |= d["mask"]

        ax[i, 0].imshow(clipped, cmap="magma");  ax[i, 0].set_title(f"t={t}  clipped NPC (watershed input)")
        ax[i, 1].imshow(_norm(npc), cmap="gray")
        ax[i, 1].imshow(np.ma.masked_where(~accepted, accepted), cmap="autumn", alpha=0.5)
        ax[i, 1].set_title("accepted droplet interiors")
        ax[i, 2].imshow(_norm(npc), cmap="gray")
        for d in droplets:
            cy, cx = d["centroid"]
            ax[i, 2].plot(cx, cy, "c+", ms=10)
        ax[i, 2].set_title(f"{len(droplets)} droplets accepted")
        for a in ax[i]:
            a.axis("off")
    plt.tight_layout(); plt.show()

In [ ]:
# ── Stage 2 Runner ─────────────────────────────────────────────────────────────
validate_watershed(hyperstack, t_range=list(range(6, min(10, T))), z=8)

---
## Stage 3 — NPC puncta label verification

Confirm NPC labels lie on the envelope ring — not deep inside the nucleus (antibody artifact) and not at the droplet wall (v6-era error). Uses the **raw** NPC plane.

In [ ]:
def verify_npc_labels(hyperstack, t: int, z: int, n_droplets: int = 5) -> None:
    """STAGE 3 — per sampled droplet: raw NPC / nucleus anchor / annular zone / puncta / overlay."""
    npc = extract_plane(hyperstack, t, z, CH_NPC)   # RAW
    nls = extract_plane(hyperstack, t, z, CH_NLS)
    droplets = [d for d in detect_droplets_npc_watershed(npc)]
    if not droplets:
        print(f"No droplets at t={t}, z={z}."); return

    # prefer droplets that actually have a nucleus to anchor to
    with_nuc = []
    for d in droplets:
        r0, c0, r1, c1 = d["bbox"]
        nuc, _ = detect_nucleus_adaptive(nls[r0:r1, c0:c1], d["mask"][r0:r1, c0:c1])
        if nuc.sum() > 0:
            with_nuc.append((d, nuc))
    if not with_nuc:
        print(f"No nucleated droplets at t={t}, z={z} — try a later timepoint."); return

    sample = [with_nuc[i] for i in rng.choice(len(with_nuc), size=min(n_droplets, len(with_nuc)), replace=False)]

    fig, ax = plt.subplots(len(sample), 5, figsize=(20, 4 * len(sample)))
    if len(sample) == 1:
        ax = ax[None, :]
    selem = morphology.disk(NPC_MARGIN_PX)
    for i, (d, nuc) in enumerate(sample):
        r0, c0, r1, c1 = d["bbox"]
        dm   = d["mask"][r0:r1, c0:c1]
        npcc = npc[r0:r1, c0:c1]
        zone = morphology.binary_dilation(nuc, selem) & ~morphology.binary_erosion(nuc, selem)
        puncta = detect_npc_puncta(npcc, nuc, dm)

        ax[i, 0].imshow(_norm(npcc), cmap="gray");      ax[i, 0].set_title("Raw NPC crop")
        ax[i, 1].imshow(nuc, cmap="gray");              ax[i, 1].set_title("Nucleus anchor")
        ax[i, 2].imshow(zone, cmap="gray");             ax[i, 2].set_title(f"Annular zone ±{NPC_MARGIN_PX}px")
        ax[i, 3].imshow(puncta, cmap="gray");           ax[i, 3].set_title(f"NPC puncta ({int(puncta.sum())}px)")
        ax[i, 4].imshow(_norm(npcc), cmap="gray")
        ax[i, 4].imshow(np.ma.masked_where(~puncta, puncta), cmap="spring", alpha=0.9)
        ax[i, 4].set_title("Overlay")
        for a in ax[i]:
            a.axis("off")
    plt.tight_layout(); plt.show()

In [ ]:
# ── Stage 3 Runner ─────────────────────────────────────────────────────────────
verify_npc_labels(hyperstack, t=5, z=8, n_droplets=5)   # avoid t=0–1 (NLS too faint)

---
## Stage 4 — Nucleus threshold sweep

Adaptive + Otsu fallback across the whole timecourse. Expected: more `none`/Otsu at early timepoints (faint, unsealed), stable larger nuclei late. The summary flags empty and runaway detections; the gallery shows every flagged droplet.

In [ ]:
def nucleus_threshold_sweep(hyperstack, t_range: list[int], z: int) -> None:
    """STAGE 4 — per-timepoint method mix + area fractions, plus a flagged-droplet gallery."""
    rows, flagged = [], []
    for t in t_range:
        nls = extract_plane(hyperstack, t, z, CH_NLS)
        npc = extract_plane(hyperstack, t, z, CH_NPC)
        droplets = detect_droplets_npc_watershed(npc)
        methods = {"adaptive": 0, "otsu": 0, "none": 0}
        fracs = []
        for d in droplets:
            r0, c0, r1, c1 = d["bbox"]
            dm = d["mask"][r0:r1, c0:c1]
            nuc, method = detect_nucleus_adaptive(nls[r0:r1, c0:c1], dm)
            methods[method] += 1
            frac = nuc.sum() / max(1, dm.sum())
            fracs.append(frac)
            # flag only implausible *positive* detections (none is expected early)
            if method != "none" and (frac >= NUCLEUS_MAX_FRACTION * 0.95):
                flagged.append((t, nls[r0:r1, c0:c1].copy(), nuc.copy(), method, frac))
        rows.append((t, len(droplets), methods, float(np.mean(fracs)) if fracs else 0.0))

    ts = [r[0] for r in rows]
    fig, ax = plt.subplots(1, 2, figsize=(15, 4))
    width = 0.6
    adapt = [r[2]["adaptive"] for r in rows]
    otsu  = [r[2]["otsu"] for r in rows]
    none_ = [r[2]["none"] for r in rows]
    ax[0].bar(ts, adapt, width, label="adaptive")
    ax[0].bar(ts, otsu, width, bottom=adapt, label="otsu")
    ax[0].bar(ts, none_, width, bottom=[a + o for a, o in zip(adapt, otsu)], label="none")
    ax[0].set_xlabel("timepoint"); ax[0].set_ylabel("droplets"); ax[0].set_title("Nucleus method mix"); ax[0].legend()
    ax[1].plot(ts, [r[3] for r in rows], "o-")
    ax[1].axhline(NUCLEUS_MAX_FRACTION, color="r", ls="--", label="max gate")
    ax[1].set_xlabel("timepoint"); ax[1].set_ylabel("mean nucleus/droplet fraction")
    ax[1].set_title("Mean nucleus area fraction"); ax[1].legend()
    plt.tight_layout(); plt.show()

    print("t  ndroplets  adaptive/otsu/none  mean_frac")
    for t, n, m, f in rows:
        print(f"{t:>2}  {n:>8}   {m['adaptive']}/{m['otsu']}/{m['none']:<6}  {f:.3f}")

    if flagged:
        k = min(len(flagged), 8)
        fig, ax = plt.subplots(2, k, figsize=(3 * k, 6))
        if k == 1:
            ax = ax[:, None]
        for j in range(k):
            t, crop, nuc, method, frac = flagged[j]
            ax[0, j].imshow(_norm(crop), cmap="gray"); ax[0, j].set_title(f"t={t} {method}\nfrac={frac:.2f}")
            ax[1, j].imshow(nuc, cmap="gray")
            ax[0, j].axis("off"); ax[1, j].axis("off")
        plt.suptitle("Flagged: implausibly large nucleus detections"); plt.tight_layout(); plt.show()
    else:
        print("\nNo runaway nucleus detections flagged.")

In [ ]:
# ── Stage 4 Runner ─────────────────────────────────────────────────────────────
nucleus_threshold_sweep(hyperstack, t_range=list(range(0, T)), z=8)

---
## Stage 5 — Patch generation with inline QC

> **Only run after Stages 1–4 pass visual inspection.**

For every valid `(t, z)`: full pipeline → `(4, H, W)` label stack → patches centred on each droplet centroid, saved as `.npz` (input `(3,H,W)` float32 + label `(4,H,W)` uint8). A random `N_PREVIEW` sample is shown after generation.

Also writes a **binary nucleus-mask hyperstack TIFF** (`(T, Z, Y, X)` uint8) of all detected nuclei — for ROI generation applied back to the original images for quantification (not for direct measurement), per the standard pipeline output.

In [ ]:
def _safe_crop(arr, cy, cx, size):
    """Centred crop with zero-padding when the window exceeds the image."""
    half = size // 2
    H, W = arr.shape[-2:]
    r0, c0 = int(round(cy)) - half, int(round(cx)) - half
    r1, c1 = r0 + size, c0 + size
    pr0, pc0 = max(0, -r0), max(0, -c0)
    sr0, sc0 = max(0, r0), max(0, c0)
    sr1, sc1 = min(H, r1), min(W, c1)
    if arr.ndim == 2:
        out = np.zeros((size, size), dtype=arr.dtype)
        out[pr0:pr0 + (sr1 - sr0), pc0:pc0 + (sc1 - sc0)] = arr[sr0:sr1, sc0:sc1]
    else:
        out = np.zeros((arr.shape[0], size, size), dtype=arr.dtype)
        out[:, pr0:pr0 + (sr1 - sr0), pc0:pc0 + (sc1 - sc0)] = arr[:, sr0:sr1, sc0:sc1]
    return out


def generate_patches_with_qc(hyperstack, t_range, z_range,
                             patch_size=PATCH_SIZE, output_dir=str(OUTPUT_DIR),
                             n_preview=N_PREVIEW) -> None:
    """STAGE 5 — generate patch pairs + nucleus-mask hyperstack TIFF, then preview."""
    out = Path(output_dir); out.mkdir(parents=True, exist_ok=True)
    T_, Z_, _, Y_, X_ = hyperstack.shape
    nucleus_hyperstack = np.zeros((T_, Z_, Y_, X_), dtype=np.uint8)
    saved = []

    for t in t_range:
        for z in z_range:
            nls = extract_plane(hyperstack, t, z, CH_NLS)
            npc = extract_plane(hyperstack, t, z, CH_NPC)   # raw, re-pulled
            mem = extract_plane(hyperstack, t, z, CH_MEMBRANE)
            droplets = detect_droplets_npc_watershed(npc)
            if not droplets:
                continue

            nuc_masks, npc_masks = [], []
            for d in droplets:
                r0, c0, r1, c1 = d["bbox"]
                dm = d["mask"][r0:r1, c0:c1]
                nuc, _ = detect_nucleus_adaptive(nls[r0:r1, c0:c1], dm)
                pun = detect_npc_puncta(npc[r0:r1, c0:c1], nuc, dm)
                nuc_masks.append(nuc); npc_masks.append(pun)

            stack = build_label_stack(nls.shape, droplets, nuc_masks, npc_masks)
            nucleus_hyperstack[t, z] = stack[CLASS_NUCLEUS]   # accumulate for TIFF

            inp = np.stack([nls, npc, mem]).astype(np.float32)
            for di, d in enumerate(droplets):
                cy, cx = d["centroid"]
                ip = _safe_crop(inp, cy, cx, patch_size)
                lp = _safe_crop(stack, cy, cx, patch_size)
                fn = out / f"patch_t{t:02d}_z{z:02d}_d{di:04d}.npz"
                np.savez_compressed(fn, input=ip, label=lp)
                saved.append(fn)

    tiff_path = out / "nucleus_mask_hyperstack.tif"
    tifffile.imwrite(str(tiff_path), nucleus_hyperstack,
                     imagej=True, metadata={"axes": "TZYX"})
    print(f"Saved {len(saved)} patches → {out}")
    print(f"Saved nucleus-mask hyperstack → {tiff_path}  shape={nucleus_hyperstack.shape}")

    if not saved:
        return
    pick = [saved[i] for i in rng.choice(len(saved), size=min(n_preview, len(saved)), replace=False)]
    fig, ax = plt.subplots(2, len(pick), figsize=(3 * len(pick), 6))
    if len(pick) == 1:
        ax = ax[:, None]
    for j, fn in enumerate(pick):
        dat = np.load(fn)
        ip, lp = dat["input"], dat["label"]
        ax[0, j].imshow(_norm(ip[0]), cmap="gray"); ax[0, j].set_title(fn.name.replace("patch_", ""), fontsize=7)
        ax[1, j].imshow(collapse_to_integer(lp), cmap=_LABEL_CMAP, vmin=0, vmax=3)
        ax[0, j].axis("off"); ax[1, j].axis("off")
    plt.suptitle("Patch QC: NLS input (top) · 4-class label (bottom)"); plt.tight_layout(); plt.show()

In [ ]:
# ── Stage 5 Runner ─────────────────────────────────────────────────────────────
# Run ONLY after Stages 1–4 look correct.
generate_patches_with_qc(
    hyperstack,
    t_range=list(range(0, T)),
    z_range=list(range(Z_FLOOR, Z)),
    patch_size=PATCH_SIZE,
    output_dir=str(OUTPUT_DIR),
    n_preview=N_PREVIEW,
)